### Установка зависимостей

In [1]:
import pandas as pd
import json
import requests
import os
import time

In [2]:
pip install --upgrade numexpr bottleneck


   ---------------------------------------- 0.0/157.6 kB ? eta -:--:--
   ------- -------------------------------- 30.7/157.6 kB 1.3 MB/s eta 0:00:01
   ----------------- --------------------- 71.7/157.6 kB 975.2 kB/s eta 0:00:01
   ---------------------------------------- 157.6/157.6 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/113.5 kB ? eta -:--:--
   ---------------------------------------- 113.5/113.5 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: numexpr
    Found existing installation: numexpr 2.8.7
    Uninstalling numexpr-2.8.7:
      Successfully uninstalled numexpr-2.8.7
  Attempting uninstall: bottleneck
    Found existing installation: Bottleneck 1.3.7
    Uninstalling Bottleneck-1.3.7:
      Successfully uninstalled Bottleneck-1.3.7


In [7]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


## IMDb dataset (TSV)
https://datasets.imdbws.com/

Состоит из 7 файлов:

- title.akas.tsv  
- title.basics.tsv  
- title.crew.tsv  
- title.episode.tsv (содержит данные о сериалах и не используется в данной работе)
- title.principals.tsv  
- title.ratings.tsv  
- name.basics.tsv  

### Классификация характеристик


Категориальные характеристики:

- genres (title.basics) — жанровый микс (Action, Drama, Comedy и т.д.).  
- titleType (title.basics) — формат/тип контента (movie, tvSeries)
- isAdult (title.basics) — бинарная категория возрастного ценза (0 — общедоступный, 1 — 18+).
- category (title.principals) — роль члена съемочной группы (director, actor, actress, writer).
- region / language (title.akas) — географический и языковой регион релиза.
  
Числовые характеристики:

- averageRating (title.ratings) — взвешенный средний зрительский рейтинг (целевая переменная успеха, от 1.0 до 10.0).  
- numVotes (title.ratings) — количество проголосовавших пользователей (показатель охвата и интереса аудитории).  
- runtimeMinutes (title.basics) — хронометраж фильма в минутах.    
  
Характеристики времени:

- startYear (title.basics) — год выхода фильма в прокат (YYYY).  
- birthYear / deathYear (name.basics) — год рождения/смерти участников съемочной группы (для расчета возраста режиссера/актера на момент выхода фильма).

### Сбор датасета

In [5]:

# title.basics
df_basics = pd.read_csv(
    r"C:\Users\tovst\Downloads\Курсовая работа\data\imdb\title.basics.tsv", 
    sep='\t', 
    na_values='\\N', 
    usecols=['tconst', 'titleType', 'primaryTitle', 'isAdult', 'startYear', 'runtimeMinutes', 'genres'],
    low_memory=False
)
# только полнометражные фильмы 2010-2025 годов
df_basics_range = df_basics[ 
    (df_basics['titleType'] == 'movie') & 
    (df_basics['startYear'] >= 2010) & 
    (df_basics['startYear'] <= 2025)
].copy()
del df_basics # Освобождаем оперативку от исходного большого датасета

# Загружаем данные о рейтинге фильмов title.ratings
df_ratings = pd.read_csv(r"C:\Users\tovst\Downloads\Курсовая работа\data\imdb\title.ratings.tsv", sep='\t', na_values='\\N')

# объединяем базовые данные и оценки по ключу tconst
imdb_merged = pd.merge(df_basics_range, df_ratings, on='tconst', how='inner')
df_ratings

# Загружаем данные о съемочной группе
df_principals = pd.read_csv(r"C:\Users\tovst\Downloads\Курсовая работа\data\imdb\title.principals.tsv", sep='\t',  na_values='\\N', usecols=['tconst', 'category'])
principals_agg = df_principals.pivot_table(index='tconst', columns='category', aggfunc='size', fill_value=0).reset_index()
del df_principals

# Переименовываем колонки
rename_dict = {}
if 'actor' in principals_agg.columns: rename_dict['actor'] = 'num_actors'
if 'actress' in principals_agg.columns: rename_dict['actress'] = 'num_actresses'
if 'director' in principals_agg.columns: rename_dict['director'] = 'num_directors'
principals_agg = principals_agg.rename(columns=rename_dict)

# Оставляем только нужные колонки для слияния
cols_to_merge = ['tconst'] + [col for col in ['num_actors', 'num_actresses', 'num_directors'] if col in principals_agg.columns]
imdb_merged = pd.merge(imdb_merged, principals_agg[cols_to_merge], on='tconst', how='left')

# Заполняем пропуски нулями
for col in ['num_actors', 'num_actresses', 'num_directors']:
    if col in imdb_merged.columns:
        imdb_merged[col] = imdb_merged[col].fillna(0).astype(int)
df_crew=pd.read_csv(r"C:\Users\tovst\Downloads\Курсовая работа\data\imdb\title.crew.tsv", sep='\t',  na_values='\\N')
imdb_merged=pd.merge(imdb_merged, df_crew, on='tconst', how='inner')
del df_crew

# данные о локализациях и региональных вариантах наименований фильмов
df_akas=pd.read_csv(r"C:\Users\tovst\Downloads\Курсовая работа\data\imdb\title.akas.tsv" , sep='\t',  na_values='\\N', usecols=['titleId', 'region', 'language', 'isOriginalTitle'])
region_agg = df_akas.groupby('titleId')['region'].count().reset_index()
region_agg = region_agg.rename(columns={'titleId': 'tconst', 'region': 'release_regions_count'})
imdb_merged = pd.merge(imdb_merged, region_agg, on='tconst', how='left')
del df_akas

# Приводим типы данных
imdb_merged['runtimeMinutes'] = pd.to_numeric(imdb_merged['runtimeMinutes'], errors='coerce')
imdb_merged['startYear'] = imdb_merged['startYear'].astype(int)

print(f"\n Успешно объединено проектов: {len(imdb_merged):,}")
print(f"Количество пропущенных значений в рейтингах: {imdb_merged['averageRating'].isnull().sum()}")

display(imdb_merged.head())


 Успешно объединено проектов: 160,825
Количество пропущенных значений в рейтингах: 0


,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,num_actors,num_actresses,num_directors,directors,writers,release_regions_count
0,tt0062336,movie,The Tango of the Widower and Its Distorting Mi...,0,2020,70.0,"Drama,Mystery",6.3,300,6,4,1,nm0749914,"nm0749914,nm1146177",10.0
1,tt0065414,movie,Appunti per un romanzo sull'immondezza,0,2022,85.0,Documentary,6.9,41,0,0,1,nm0001596,NaN,1.0
2,tt0069049,movie,The Other Side of the Wind,0,2018,122.0,Drama,6.6,8641,6,4,1,nm0000080,"nm0000080,nm0462648",27.0
3,tt0070596,movie,Socialist Realism,0,2023,78.0,"Drama,History",6.8,244,10,0,1,nm0749914,"nm0667610,nm0749914,nm0765384",6.0
4,tt0088837,movie,"Hamara Shahar - Bombay, Our City",0,2010,82.0,Documentary,8.2,64,0,0,1,nm0666674,NaN,4.0


In [25]:
# Путь для сохранения готового промежуточного файла
output_dir = r"C:\Users\tovst\Downloads\Курсовая работа\data\imdb\\"
os.makedirs(output_dir, exist_ok=True)
file_path = os.path.join(output_dir, "imdb_merged_2010_2025.parquet")

#Сохранение в формате Parquet 
imdb_merged.to_parquet("C:/Users/tovst/Downloads/Курсовая работа/data/tmdb_mass_dataset_2010_2025.parquet", index=False)
print(f"Датасет успешно сохранен на диск: {file_path}")
print(f"Размер сохраненного файла: {os.path.getsize(file_path) / (1024*1024):.2f} МБ")

Датасет успешно сохранен на диск: C:\Users\tovst\Downloads\Курсовая работа\data\imdb\\imdb_merged_2010_2025.parquet
Размер сохраненного файла: 8.02 МБ


Справочник сущностей id-имя творческой группы

In [28]:
data_dir=r"C:\Users\tovst\Downloads\Курсовая работа\data\imdb\\"
df_p_names=pd.read_csv(data_dir + 'title.principals.tsv', sep='\t', na_values='\\N', usecols=['tconst', 'nconst', 'category'])
df_p_names = df_p_names[df_p_names['tconst'].isin(imdb_merged['tconst'])].copy() # строки, которые относятся к нашим фильмам из imdb_merged
unique_nconsts = df_p_names['nconst'].unique() # ищем уникальные строки

# Загружаем справочник имен name.basics, для тех людей, которые есть в отфильтрованном списке unique_nconsts:
df_names = pd.read_csv(data_dir+'name.basics.tsv',sep='\t', na_values='\\N', usecols=['nconst', 'primaryName', 'birthYear', 'primaryProfession'])
df_names = df_names[df_names['nconst'].isin(unique_nconsts)].copy()

# Справочник ID-Имя
actors_directors_lookup = df_names[['nconst', 'primaryName', 'birthYear', 'primaryProfession']].copy()
print(f"Уникальных персон в фильмах: {len(actors_directors_lookup):,}")

# Создание дополнительного поля 'director names' для imdb_merged
df_directors = df_p_names[df_p_names['category'] == 'director'][['tconst', 'nconst']].copy()

# Объединяем со справочником имен
df_directors_named = pd.merge(df_directors, actors_directors_lookup, on='nconst', how='inner')

if not df_directors_named.empty and 'primaryName' in df_directors_named.columns:
    directors_agg = df_directors_named.groupby('tconst')['primaryName'].agg(lambda x: ', '.join(x.dropna().astype(str))).reset_index()
    directors_agg = directors_agg.rename(columns={'primaryName': 'director_names'})
    
    # Если колонка уже есть в imdb_merged, удаляем её во избежание конфликта MergeError
    if 'director_names' in imdb_merged.columns:
        imdb_merged = imdb_merged.drop(columns=['director_names'])
        
    # Соединение с imdb_merged
    imdb_merged = pd.merge(imdb_merged, directors_agg, on='tconst', how='left')
else:
    if 'director_names' not in imdb_merged.columns:
        imdb_merged['director_names'] = 'Unknown'

# Заполняем пропуски, если для каких-то фильмов режиссер не нашелся
imdb_merged['director_names'] = imdb_merged['director_names'].fillna('Unknown')

print(f"Обработано фильмов с режиссерами: {(imdb_merged['director_names'] != 'Unknown').sum():,}")
display(imdb_merged[['tconst', 'primaryTitle', 'startYear', 'director_names']].head(10))

Уникальных персон в фильмах: 989,194
Обработано фильмов с режиссерами: 157,157


,tconst,primaryTitle,startYear,director_names
0,tt0062336,The Tango of the Widower and Its Distorting Mi...,2020,Raúl Ruiz
1,tt0065414,Appunti per un romanzo sull'immondezza,2022,Pier Paolo Pasolini
2,tt0069049,The Other Side of the Wind,2018,Orson Welles
3,tt0070596,Socialist Realism,2023,Raúl Ruiz
4,tt0088837,"Hamara Shahar - Bombay, Our City",2010,Anand Patwardhan
5,tt0097767,Loading Ludwig,2022,Mara Mattuschka
6,tt0100275,The Wandering Soap Opera,2017,"Raúl Ruiz, Valeria Sarmiento"
7,tt0107583,The Misfits - 30 Years of Fluxus,2011,Lars Movin
8,tt0111596,Vazir,2015,Sanjay D Rawal
9,tt0112502,Bigfoot,2017,Mc Jones


Анализ датасета

In [19]:
imdb_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 160825 entries, 0 to 160824
Data columns (total 18 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   tconst                 160825 non-null  str    
 1   titleType              160825 non-null  str    
 2   primaryTitle           160823 non-null  str    
 3   isAdult                160825 non-null  int64  
 4   startYear              160825 non-null  int32  
 5   runtimeMinutes         149146 non-null  float64
 6   genres                 158313 non-null  str    
 7   averageRating          160825 non-null  float64
 8   numVotes               160825 non-null  int64  
 9   num_actors             160825 non-null  int32  
 10  num_actresses          160825 non-null  int32  
 11  num_directors          160825 non-null  int32  
 12  directors              158413 non-null  str    
 13  writers                137328 non-null  str    
 14  release_regions_count  160693 non-null  float64

In [20]:
display(imdb_merged.describe())

,isAdult,startYear,runtimeMinutes,averageRating,numVotes,num_actors,num_actresses,num_directors,release_regions_count
count,160825.000000,160825.000000,149146.000000,160825.000000,1.608250e+05,160825.000000,160825.000000,160825.000000,160693.000000
mean,0.000958,2018.008096,95.058721,6.232135,3.658630e+03,4.317388,2.546663,1.095122,8.966053
std,0.030930,4.507221,194.567569,1.515430,3.522619e+04,3.061385,2.183207,0.442981,8.927998
min,0.000000,2010.000000,4.000000,1.000000,5.000000e+00,0.000000,0.000000,0.000000,0.000000
25%,0.000000,2014.000000,80.000000,5.300000,2.000000e+01,1.000000,0.000000,1.000000,3.000000
50%,0.000000,2018.000000,91.000000,6.400000,6.800000e+01,5.000000,2.000000,1.000000,8.000000
75%,0.000000,2022.000000,105.000000,7.300000,3.620000e+02,7.000000,4.000000,1.000000,10.000000
max,1.000000,2025.000000,51420.000000,10.000000,2.876060e+06,34.000000,35.000000,4.000000,148.000000


In [21]:
display(imdb_merged.select_dtypes(include=['number']).corr())

,isAdult,startYear,runtimeMinutes,averageRating,numVotes,num_actors,num_actresses,num_directors,release_regions_count
isAdult,1.000000,-0.004159,0.001164,0.006083,-0.003179,-0.016278,0.011033,-0.011640,-0.023738
startYear,-0.004159,1.000000,0.006008,0.005641,-0.033365,-0.014903,0.028090,-0.001426,0.033900
runtimeMinutes,0.001164,0.006008,1.000000,-0.002339,0.012618,0.028600,0.015408,0.004420,0.026537
averageRating,0.006083,0.005641,-0.002339,1.000000,0.039364,-0.269648,-0.285872,-0.001151,-0.071158
numVotes,-0.003179,-0.033365,0.012618,0.039364,1.000000,0.082994,0.040960,-0.001210,0.462999
num_actors,-0.016278,-0.014903,0.028600,-0.269648,0.082994,1.000000,0.327691,0.012121,0.227275
num_actresses,0.011033,0.028090,0.015408,-0.285872,0.040960,0.327691,1.000000,0.009677,0.211879
num_directors,-0.011640,-0.001426,0.004420,-0.001151,-0.001210,0.012121,0.009677,1.000000,0.004306
release_regions_count,-0.023738,0.033900,0.026537,-0.071158,0.462999,0.227275,0.211879,0.004306,1.000000


In [35]:
print("\n2. ОПРЕДЕЛЕНИЕ ТИПОВ ШКАЛ И АНАЛИТИЧЕСКИХ РОЛЕЙ АТРИБУТОВ")

# Формируем таблицу метаданных для отчета
scales_roles_df = pd.DataFrame({
    'Атрибут (Поле)': ['startYear', 'runtimeMinutes', 'averageRating', 'numVotes', 'genres', 'isAdult', 'release_regions_count'],
    'Тип шкалы': [
        'Интервальная (абсолютная)', 
        'Абсолютная (метрическая)', 
        'Порядковая / Интервальная', 
        'Абсолютная (метрическая)', 
        'Номинальная (шкала наименований)', 
        'Бинэрная (абсолютная)', 
        'Абсолютная (метрическая)'
    ],
    'Аналитическая роль': [
        'Временной признак / Фильтрация', 
        'Факторная переменная (длительность)', 
        'Целевая переменная (успех/качество)', 
        'Факторная переменная (охват/популярность)', 
        'Категориальный признак (группировка)', 
        'Фильтр качества / Ценз', 
        'Факторная переменная (масштаб проката)'
    ]
})
display(scales_roles_df)


2. ОПРЕДЕЛЕНИЕ ТИПОВ ШКАЛ И АНАЛИТИЧЕСКИХ РОЛЕЙ АТРИБУТОВ


,Атрибут (Поле),Тип шкалы,Аналитическая роль
0,startYear,Интервальная (абсолютная),Временной признак / Фильтрация
1,runtimeMinutes,Абсолютная (метрическая),Факторная переменная (длительность)
2,averageRating,Порядковая / Интервальная,Целевая переменная (успех/качество)
3,numVotes,Абсолютная (метрическая),Факторная переменная (охват/популярность)
4,genres,Номинальная (шкала наименований),Категориальный признак (группировка)
5,isAdult,Бинэрная (абсолютная),Фильтр качества / Ценз
6,release_regions_count,Абсолютная (метрическая),Факторная переменная (масштаб проката)


## TMDB (json-csv)

В официальной документации TMDB отдаёт данные через API в формате JSON (https://www.omdbapi.com/) , но для удобства анализа сохранено их таблицы (CSV: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata?resource=download).

In [43]:
data_dir = r"C:\Users\tovst\Downloads\Курсовая работа\data\tmdb\\"
tmdb_credits_p1 = pd.read_csv(data_dir+"tmdb_5000_credits.csv")
tmdb_movies_p2 = pd.read_csv(data_dir+"tmdb_5000_movies.csv")

print(f"Размер первой таблицы: {tmdb_credits_p1.shape}")
print(f"Размер второй таблицы: {tmdb_movies_p2.shape}")

tmdb_movies = pd.merge(tmdb_credits_p1,tmdb_movies_p2, left_on='movie_id', right_on='id', how='inner')

if 'title_x' in tmdb_movies.columns:
    tmdb_movies = tmdb_movies.rename(columns={'title_x': 'title'})
if 'title_y' in tmdb_movies.columns:
    tmdb_movies = tmdb_movies.drop(columns=['title_y'])
print("Поля в tmdb_movies:", tmdb_movies.columns.tolist())
tmdb_movies.head(10)

Размер первой таблицы: (4803, 4)
Размер второй таблицы: (4803, 20)
Поля в tmdb_movies: ['movie_id', 'title', 'cast', 'crew', 'budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'vote_average', 'vote_count']


,movie_id,title,cast,crew,budget,genres,homepage,id,keywords,original_language,...,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,...,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,7.2,11800
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,...,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",6.9,4500
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,...,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,6.3,4466
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,...,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,7.6,9106
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,...,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",6.1,2124
5,559,Spider-Man 3,"[{""cast_id"": 30, ""character"": ""Peter Parker / ...","[{""credit_id"": ""52fe4252c3a36847f80151a5"", ""de...",258000000,"[{""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""na...",http://www.sonypictures.com/movies/spider-man3/,559,"[{""id"": 851, ""name"": ""dual identity""}, {""id"": ...",en,...,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-01,890871626,139.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,The battle within.,5.9,3576
6,38757,Tangled,"[{""cast_id"": 34, ""character"": ""Flynn Rider (vo...","[{""credit_id"": ""52fe46db9251416c91062101"", ""de...",260000000,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 10751...",http://disney.go.com/disneypictures/tangled/,38757,"[{""id"": 1562, ""name"": ""hostage""}, {""id"": 2343,...",en,...,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {

### Парсинг json-полей

In [45]:
def safe_parse_json(json_str):
    if pd.isna(json_str):
        return []
    try:
        return json.loads(json_str.replace("'", '"'))
    except Exception:
        try:
            # Запасной вариант парсинга, если стандартный дал сбой
            import ast
            return ast.literal_eval(json_str)
        except Exception:
            return []

# 1. парсинг жанров (genres): 
# в TMDB это список словарей вида [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]
if 'genres' in tmdb_movies.columns:
    # Парсим JSON в списки словарей и собираем имена жанров через запятую
    tmdb_movies['genres_parsed'] = tmdb_movies['genres'].apply(safe_parse_json)
    tmdb_movies['genres_list'] = tmdb_movies['genres_parsed'].apply(
        lambda x: [item['name'] for item in x if isinstance(item, dict) and 'name' in item]
    )
    tmdb_movies['genres_str'] = tmdb_movies['genres_list'].apply(lambda x: ', '.join(x))

# 2. парсинг актеров (cast):
# Извлекаем имена главных актеров  (топ-5)
if 'cast' in tmdb_movies.columns:
    tmdb_movies['cast_parsed'] = tmdb_movies['cast'].apply(safe_parse_json)
    tmdb_movies['top_actors'] = tmdb_movies['cast_parsed'].apply(
        lambda x: ', '.join([item['name'] for item in x[:5] if isinstance(item, dict) and 'name' in item]) 
        if isinstance(x, list) else ''
    )

display(tmdb_movies[['title', 'genres_str', 'top_actors', 'budget', 'revenue']].head(3))

,title,genres_str,top_actors,budget,revenue
0,Avatar,"Action, Adventure, Fantasy, Science Fiction","Sam Worthington, Zoe Saldana, Sigourney Weaver...",237000000,2787965087
1,Pirates of the Caribbean: At World's End,"Adventure, Fantasy, Action","Johnny Depp, Orlando Bloom, Keira Knightley, S...",300000000,961000000
2,Spectre,"Action, Adventure, Crime","Daniel Craig, Christoph Waltz, Léa Seydoux, Ra...",245000000,880674609


In [47]:
tmdb_df_to_filter = tmdb_movies
tmdb_df_to_filter['release_date_dt'] = pd.to_datetime(tmdb_df_to_filter['release_date'], errors='coerce')
tmdb_df_to_filter['release_year'] = tmdb_df_to_filter['release_date_dt'].dt.year
tmdb_filtered = tmdb_df_to_filter[
    (tmdb_df_to_filter['release_year'] >= 2010) & 
    (tmdb_df_to_filter['release_year'] <= 2025)
].copy()
tmdb_filtered.to_parquet(r"C:\Users\tovst\Downloads\Курсовая работа\data\tmdb\tmdb_movies_2010_2025.parquet", index=False)

In [49]:
tmdb_output_dir = r"C:\Users\tovst\Downloads\Курсовая работа\data\tmdb"
os.makedirs(tmdb_output_dir, exist_ok=True)

tmdb_parquet_path = os.path.join(tmdb_output_dir, "tmdb_2010_2025.parquet")

In [55]:
tmdb_filtered.info()

<class 'pandas.DataFrame'>
Index: 1446 entries, 2 to 4801
Data columns (total 30 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   movie_id              1446 non-null   int64         
 1   title                 1446 non-null   str           
 2   cast                  1446 non-null   str           
 3   crew                  1446 non-null   str           
 4   budget                1446 non-null   int64         
 5   genres                1446 non-null   str           
 6   homepage              832 non-null    str           
 7   id                    1446 non-null   int64         
 8   keywords              1446 non-null   str           
 9   original_language     1446 non-null   str           
 10  original_title        1446 non-null   str           
 11  overview              1443 non-null   str           
 12  popularity            1446 non-null   float64       
 13  production_companies  1446 non-nul

### Характеристики

Числовые (количественные): budget, revenue, popularity, runtime, vote_average, vote_count.  
Качественные (категориальные и текстовые): original_title, status, original_language, распарсенные списки жанров и актеров.  
Временные характеристики: Поле release_date (дата релиза), преобразованное в формат datetime, позволяющее определить временной охват датасета (диапазон лет).

In [64]:
tmdb_movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 28 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movie_id              4803 non-null   int64  
 1   title                 4803 non-null   str    
 2   cast                  4803 non-null   str    
 3   crew                  4803 non-null   str    
 4   budget                4803 non-null   int64  
 5   genres                4803 non-null   str    
 6   homepage              1712 non-null   str    
 7   id                    4803 non-null   int64  
 8   keywords              4803 non-null   str    
 9   original_language     4803 non-null   str    
 10  original_title        4803 non-null   str    
 11  overview              4800 non-null   str    
 12  popularity            4803 non-null   float64
 13  production_companies  4803 non-null   str    
 14  production_countries  4803 non-null   str    
 15  release_date          4802 non-n

In [68]:
tmdb_movies.describe()

,movie_id,budget,id,popularity,revenue,runtime,vote_average,vote_count
count,4803.000000,4.803000e+03,4803.000000,4803.000000,4.803000e+03,4801.000000,4803.000000,4803.000000
mean,57165.484281,2.904504e+07,57165.484281,21.492301,8.226064e+07,106.875859,6.092172,690.217989
std,88694.614033,4.072239e+07,88694.614033,31.816650,1.628571e+08,22.611935,1.194612,1234.585891
min,5.000000,0.000000e+00,5.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
25%,9014.500000,7.900000e+05,9014.500000,4.668070,0.000000e+00,94.000000,5.600000,54.000000
50%,14629.000000,1.500000e+07,14629.000000,12.921594,1.917000e+07,103.000000,6.200000,235.000000
75%,58610.500000,4.000000e+07,58610.500000,28.313505,9.291719e+07,118.000000,6.800000,737.000000
max,459488.000000,3.800000e+08,459488.000000,875.581305,2.787965e+09,338.000000,10.000000,13752.000000


Количество и процент пропусков

In [73]:
print('Количество пропусков: ', tmdb_movies.isnull().sum())
print('\nПроцент пропусков: ', (tmdb_movies.isnull().mean() * 100).round(2))

Количество пропусков:  movie_id                   0
title                      0
cast                       0
crew                       0
budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
vote_average               0
vote_count                 0
genres_parsed              0
genres_list                0
genres_str                 0
cast_parsed                0
top_actors                 0
dtype: int64

Процент пропусков:  movie_id                 0.00
title                    0.00
cast                     0.00
crew                     0.00
budget     

Анализ временных характеристик

In [75]:
if 'release_date' in tmdb_movies.columns:
    # Превращаем дату релиза в формат datetime для анализа
    tmdb_movies['release_date_dt'] = pd.to_datetime(tmdb_movies['release_date'], errors='coerce')
    min_date = tmdb_movies['release_date_dt'].min()
    max_date = tmdb_movies['release_date_dt'].max()
    print(f"Временной диапазон данных (релиз фильмов): с {str(min_date)[:10]} по {str(max_date)[:10]}")

Временной диапазон данных (релиз фильмов): с 1916-09-04 по 2017-02-03


*В дальнейшем необходимо будет урезать до диапазона (2010-2017)*

### Классификация атрибутов по шкалам измерения:

1. **Метрические и абсолютные шкалы (Количественные финансовые агрегаты):**
   * *Атрибуты:* `budget` (производственный бюджет), `revenue` (кассовые сборы).
   * *Роль в модели:* Главные экономические показатели. Позволяют рассчитывать рентабельность фильмов (ROI) и проверять гипотезы о влиянии бюджета на кассовый успех.

2. **Относительные и рейтинговые шкалы:**
   * *Атрибуты:* `popularity` (популярность TMDB), `vote_average` (средняя оценка), `vote_count` (количество голосов), `runtime` (хронометраж).
   * *Роль в модели:* Факторные переменные для оценки зрительского интереса и качества контента.

3. **Категориальные и текстовые шкалы:**
   * *Атрибуты:* `genres_str` (распарсенные жанры), `top_actors` (ключевые актеры), `original_language` (язык).
   * *Роль в модели:* Используются для сегментации и факторного анализа (например, влияние конкретных жанров или языков на сборы).

4. **Ключевые атрибуты для слияния:**
   * Универсальный идентификатор `id` / `movie_id` и нормализованное название фильма (`title`), используемые для соединения с данными IMDb и Rotten Tomatoes.

## The Oscar Awards (XML)

In [69]:
oscar_df = pd.read_csv(r"C:\Users\tovst\Downloads\Курсовая работа\data\the_oscar_award\oscar_data.csv", sep='\t')
# Конвертация в XML
xml_path = r"C:\Users\tovst\Downloads\Курсовая работа\data\the_oscar_award\oscar_awards.xml"
oscar_df.to_xml(xml_path, index=False, parser='lxml', encoding='utf-8')
display(oscar_df.head(3))

,Ceremony,Year,Class,CanonicalCategory,Category,NomId,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation,MultifilmNomination
0,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051251,The Noose,tt0019217,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,Nickie Elkins,NaN,NaN,True
1,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051252,The Patent Leather Kid,tt0018253,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,The Patent Leather Kid,NaN,NaN,True
2,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051250a,The Last Command,tt0019071,Emil Jannings,Emil Jannings,nm0417837,True,General Dolgorucki [Grand Duke Sergius Alexander],NaN,NaN,True


### Характеристики

In [9]:
oscar_df.shape

(12014, 16)

In [11]:
oscar_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12014 entries, 0 to 12013
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Ceremony             12014 non-null  int64 
 1   Year                 12014 non-null  str   
 2   Class                12014 non-null  str   
 3   CanonicalCategory    12014 non-null  str   
 4   Category             12014 non-null  str   
 5   NomId                11482 non-null  str   
 6   Film                 10753 non-null  str   
 7   FilmId               10753 non-null  str   
 8   Name                 10829 non-null  str   
 9   Nominees             11661 non-null  str   
 10  NomineeIds           11134 non-null  str   
 11  Winner               3476 non-null   object
 12  Detail               3171 non-null   str   
 13  Note                 616 non-null    str   
 14  Citation             1183 non-null   str   
 15  MultifilmNomination  40 non-null     object
dtypes: int64(1), ob

In [13]:
oscar_df.describe()

,Ceremony
count,12014.000000
mean,50.855169
std,27.518439
min,1.000000
25%,26.000000
50%,51.000000
75%,75.000000
max,97.000000


In [17]:
print('Количество пропусков:\n ', oscar_df.isnull().sum())
print('\nПроцент пропусков: \n', (oscar_df.isnull().mean() * 100).round(2))

Количество пропусков:
  Ceremony                   0
Year                       0
Class                      0
CanonicalCategory          0
Category                   0
NomId                    532
Film                    1261
FilmId                  1261
Name                    1185
Nominees                 353
NomineeIds               880
Winner                  8538
Detail                  8843
Note                   11398
Citation               10831
MultifilmNomination    11974
dtype: int64

Процент пропусков: 
 Ceremony                0.00
Year                    0.00
Class                   0.00
CanonicalCategory       0.00
Category                0.00
NomId                   4.43
Film                   10.50
FilmId                 10.50
Name                    9.86
Nominees                2.94
NomineeIds              7.32
Winner                 71.07
Detail                 73.61
Note                   94.87
Citation               90.15
MultifilmNomination    99.67
dtype: float6

In [37]:
if 'Year' in oscar_df.columns or 'ceremony' in oscar_df.columns:
    display(oscar_df['Year'].sort_values(ascending=False).head(10))
else:
    print("Доступные колонки:", oscar_df.columns.tolist()[:5])



11876    2024
11877    2024
11878    2024
11879    2024
11880    2024
11881    2024
11882    2024
11883    2024
11884    2024
11885    2024
Name: Year, dtype: str

Классификация характеристик

- Временные характеристики:  year - год проведения церемонии  
- Качественные характеристики: Category - категория номинации,Film - название фильма (ключ для соединения с остальными датасетами), Name - имя номинанта, Winner, Detail, Note, Class - внутренний класс или группировка номинации  
- Идентификационные поля - NomId, FilmID, NomineeIds - уникальные строковые или числовые идентификаторы объектов в базе данных наград (используются для обеспечения реляционной целостности внутри самого датасета)

## Rotten tomatoes reviews и rotten tomatoes movies (text)

In [56]:
data_dir = r"C:\Users\tovst\Downloads\Курсовая работа\data\rotten_tomaroes_reviews\\"
rt_movies = pd.read_csv(data_dir + "rotten_tomatoes_movies.csv")
rt_reviews = pd.read_csv(data_dir + "rotten_tomatoes_movie_reviews.csv")


### rotten_tomatoes_movies

Датасет содержит агрегированные метаданные фильмов, оценки критиков и зрителей, а также производственные показатели.  

**Количественные характеристики:**  
- audienceScore — процент зрителей, которым понравился фильм (оценка аудитории по шкале от 0 до 100).
- tomatoMeter — индекс свежести по версии профессиональных кинокритиков (доля положительных рецензий, от 0 до 100).
- runtimeMinutes — хронометраж фильма в минутах.
- boxOffice — кассовые сборы фильма в прокате (числовой показатель).

**Качественные (категориальные и текстовые) характеристики:**
- id — уникальный строковый идентификатор фильма в системе Rotten Tomatoes.
- title — название фильма.
- rating — прокатный рейтинг возрастной цензуры (например, PG-13, R).
- ratingContents — текстовое описание причин присвоения возрастного рейтинга (например, содержание сцен насилия или ненормативной лексики).
- genre — жанры фильма (текстовая строка или список жанров).
- originalLanguage — язык оригинала фильма.
- director — режиссер (или список режиссеров).
- writer — сценарист (или авторы сценария).
- distributor — дистрибьютор фильма.
- soundMix — звуковая дорожка / формат аудио.

**Временные характеристики:**
- releaseDateTheaters — точная дата выхода фильма в театральный (кинотеатральный) прокат. Используется для фильтрации датасета под целевой временной диапазон 2010–2025 гг.
- releaseDateStreaming — дата релиза фильма на стриминговых платформах.

In [65]:
rt_movies

,id,title,audienceScore,tomatoMeter,rating,ratingContents,releaseDateTheaters,releaseDateStreaming,runtimeMinutes,genre,originalLanguage,director,writer,boxOffice,distributor,soundMix
0,space-zombie-bingo,Space Zombie Bingo!,50.0,NaN,NaN,NaN,NaN,2018-08-25,75.0,"Comedy, Horror, Sci-fi",English,George Ormrod,"George Ormrod,John Sabotta",NaN,NaN,NaN
1,the_green_grass,The Green Grass,NaN,NaN,NaN,NaN,NaN,2020-02-11,114.0,Drama,English,Tiffany Edwards,Tiffany Edwards,NaN,NaN,NaN
2,love_lies,"Love, Lies",43.0,NaN,NaN,NaN,NaN,NaN,120.0,Drama,Korean,"Park Heung-Sik,Heung-Sik Park","Ha Young-Joon,Jeon Yun-su,Song Hye-jin",NaN,NaN,NaN
3,the_sore_losers_1997,Sore Losers,60.0,NaN,NaN,NaN,NaN,2020-10-23,90.0,"Action, Mystery & thriller",English,John Michael McCarthy,John Michael McCarthy,NaN,NaN,NaN
4,dinosaur_island_2002,Dinosaur Island,70.0,NaN,NaN,NaN,NaN,2017-03-27,80.0,"Fantasy, Adventure, Animation",English,Will Meugniot,John Loy,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143253,nadia_the_secret_of_blue_water_the_motion_pict...,Nadia: The Secret of Blue Water: The Motion Pi...,14.0,NaN,NaN,NaN,2002-08-27,NaN,90.0,"Action, Adventure, Anime",Japanese,Sho Aono,Kaoru Umeno,NaN,ADV Films,NaN
143254,everyone_i_knew_and_loved,Everyone I Knew and Loved,NaN,NaN,NaN,NaN,NaN,NaN,99.0,Drama,English,Andrew Behringer,Erika Heidewald,NaN,NaN,NaN
143255,the-human-body,The Human Body,71.0,89.0,NaN,NaN,NaN,NaN,43.0,Documentary,English,Peter Georgi,Richard Dale,NaN,NaN,NaN
143256,flying_fists,Flying Fists,NaN,NaN,NaN,NaN,NaN,2006-11-21,63.0,Drama,English,Robert F. Hill,"Robert F. Hill,Basil Dickey",NaN,NaN,NaN


In [90]:
print(rt_movies['releaseDateTheaters'].sort_values(ascending=False).head(10))
print(rt_movies['releaseDateTheaters'].sort_values(ascending=True).head(10))

123693    2032-04-21
50971     2028-12-22
36508     2026-12-18
25752     2025-02-14
17044     2024-12-25
79625     2024-12-20
127276    2024-11-08
31968     2024-10-04
23408     2024-09-06
139051    2024-07-05
Name: releaseDateTheaters, dtype: str
11445     1902-10-04
107954    1903-10-17
31154     1904-03-26
60206     1905-05-29
15630     1907-12-07
48458     1908-02-01
143176    1908-07-14
102114    1909-06-10
111941    1909-07-19
5466      1909-09-02
Name: releaseDateTheaters, dtype: str


In [69]:
rt_movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 143258 entries, 0 to 143257
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    143258 non-null  str    
 1   title                 142891 non-null  str    
 2   audienceScore         73248 non-null   float64
 3   tomatoMeter           33877 non-null   float64
 4   rating                13991 non-null   str    
 5   ratingContents        13991 non-null   str    
 6   releaseDateTheaters   30773 non-null   str    
 7   releaseDateStreaming  79420 non-null   str    
 8   runtimeMinutes        129431 non-null  float64
 9   genre                 132175 non-null  str    
 10  originalLanguage      129400 non-null  str    
 11  director              139041 non-null  str    
 12  writer                90116 non-null   str    
 13  boxOffice             14743 non-null   str    
 14  distributor           23001 non-null   str    
 15  soundMix   

In [85]:
rt_movies.describe()

,audienceScore,tomatoMeter,runtimeMinutes
count,73248.000000,33877.000000,129431.000000
mean,55.674967,65.770346,93.708578
std,24.553648,28.023203,28.129175
min,0.000000,0.000000,1.000000
25%,37.000000,45.000000,84.000000
50%,57.000000,73.000000,92.000000
75%,76.000000,89.000000,103.000000
max,100.000000,100.000000,2700.000000


In [75]:
print('Количество пропусков:\n ', rt_movies.isnull().sum())
print('\nПроцент пропусков: \n', (rt_movies.isnull().mean() * 100).round(2))

Количество пропусков:
  id                           0
title                      367
audienceScore            70010
tomatoMeter             109381
rating                  129267
ratingContents          129267
releaseDateTheaters     112485
releaseDateStreaming     63838
runtimeMinutes           13827
genre                    11083
originalLanguage         13858
director                  4217
writer                   53142
boxOffice               128515
distributor             120257
soundMix                127341
dtype: int64

Процент пропусков: 
 id                       0.00
title                    0.26
audienceScore           48.87
tomatoMeter             76.35
rating                  90.23
ratingContents          90.23
releaseDateTheaters     78.52
releaseDateStreaming    44.56
runtimeMinutes           9.65
genre                    7.74
originalLanguage         9.67
director                 2.94
writer                  37.10
boxOffice               89.71
distributor             8

In [ ]:
releaseDateTheaters

### rotten_tomatoes_movie_reviews

Датасет хранит рецензии на фильмы в форматк одна строка-одна рецензия.  

**Количественные характеристики:**
- originalScore — оценка кинокритика в оригинальном формате (например, 3.5/4, 7/10), которая может быть переведена в числовой вид.

**Качественные (категориальные и текстовые) характеристики:**  
- id — идентификатор фильма, к которому относится рецензия (используется для связи с таблицей фильмов).
- reviewId — уникальный идентификатор конкретной рецензии.
- criticName — имя автора рецензии (кинокритика).
- isTopCritic — логический признак (True/False), указывающий, является ли автор ведущим/признанным критиком.
- reviewState — статус рецензии (например, fresh — положительная, rotten — отрицательная).
- publicationName — название издания, опубликовавшего рецензию (например, New York Times, Variety).
- reviewText — полный текст рецензии (ключевое поле для NLP-анализа и оценки тональности).
- scoreSentiment — общая тональность или сентимент оценки (POSITIVE, NEGATIVE и т.д.).
- reviewUrl — веб-ссылка на оригинальную публикацию рецензии.

**Временные характеристики:**
- creationDate — точная дата и время создания/публикации рецензии. Позволяет отслеживать динамику мнений критиков со временем.

In [77]:
rt_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 1444963 entries, 0 to 1444962
Data columns (total 11 columns):
 #   Column          Non-Null Count    Dtype
---  ------          --------------    -----
 0   id              1444963 non-null  str  
 1   reviewId        1444963 non-null  int64
 2   creationDate    1444963 non-null  str  
 3   criticName      1444963 non-null  str  
 4   isTopCritic     1444963 non-null  bool 
 5   originalScore   1009745 non-null  str  
 6   reviewState     1444963 non-null  str  
 7   publicatioName  1444963 non-null  str  
 8   reviewText      1375738 non-null  str  
 9   scoreSentiment  1444963 non-null  str  
 10  reviewUrl       1234038 non-null  str  
dtypes: bool(1), int64(1), str(9)
memory usage: 470.5 MB


In [81]:
print('Количество пропусков:\n ', rt_reviews.isnull().sum())
print('\nПроцент пропусков: \n', (rt_reviews.isnull().mean() * 100).round(2))

Количество пропусков:
  id                     0
reviewId               0
creationDate           0
criticName             0
isTopCritic            0
originalScore     435218
reviewState            0
publicatioName         0
reviewText         69225
scoreSentiment         0
reviewUrl         210925
dtype: int64

Процент пропусков: 
 id                 0.00
reviewId           0.00
creationDate       0.00
criticName         0.00
isTopCritic        0.00
originalScore     30.12
reviewState        0.00
publicatioName     0.00
reviewText         4.79
scoreSentiment     0.00
reviewUrl         14.60
dtype: float64


## Kinopoisk's movies reviews

Датасет представляет собой иерерхию папок с разбивкой по тональности. Каждая рецензия находтися в отдельном .txt файле

### Сбор данных в единый датафрейм

In [108]:
data_dir = r"C:\Users\tovst\Downloads\Курсовая работа\data\kinopoisk_reviews\\"

# Словарь для сопоставления названий папок с метками тональности
# (проверьте точные названия ваших папок, например: 'positive', 'negative', 'neutral')
sentiment_folders = {
    'pos': 'POSITIVE',
    'neg': 'NEGATIVE',
    'neu': 'NEUTRAL' 
}

reviews_data = []

for folder_name, sentiment_label in sentiment_folders.items():
    folder_path = os.path.join(data_dir, folder_name)
    
    if os.path.exists(folder_path):
        print(f"Обработка папки: {folder_name}...")
        for file_name in os.listdir(folder_path):
            if file_name.endswith('.txt'):
                file_path = os.path.join(folder_path, file_name)
                try:
                    # Читаем текст рецензии с кодировкой utf-8
                    with open(file_path, 'r', encoding='utf-8') as f:
                        text = f.read()
                    
                    reviews_data.append({
                        'file_name': file_name,
                        'review_text': text,
                        'sentiment': sentiment_label
                    })
                except Exception as e:
                    # На случай проблем с кодировкой пробуем cp1251
                    try:
                        with open(file_path, 'r', encoding='cp1251') as f:
                            text = f.read()
                        reviews_data.append({
                            'file_name': file_name,
                            'review_text': text,
                            'sentiment': sentiment_label
                        })
                    except Exception:
                        pass

# Создаем единый датафрейм рецензий
kp_reviews_df = pd.DataFrame(reviews_data)
print(f'\nВсего успешно загружено текстовых рецензий: {len(kp_reviews_df):,}')
display(kp_reviews_df.head(3))

Обработка папки: pos...
Обработка папки: neg...
Обработка папки: neu...

Всего успешно загружено текстовых рецензий: 131,669


,file_name,review_text,sentiment
0,1000023-0.txt,"Что делать, если любовница занимается организа...",POSITIVE
1,1000023-1.txt,Сюжет картины очень напоминает популярные амер...,POSITIVE
2,1000023-10.txt,"Ребят, я только что из кинотеатра, в полном во...",POSITIVE


## TMDB API (JSON) 
https://developer.themoviedb.org/docs/getting-started

### Выгрузка данных

In [6]:
API_BEARER_TOKEN = "API"

HEADERS = {
    "accept": "application/json",
    "Authorization": f"Bearer {API_BEARER_TOKEN}"
}

BASE_URL = "https://api.themoviedb.org/3"

def fetch_tmdb_movies_safe(start_year=2010, end_year=2025, target_pages=75):
    all_movies = []
    output_dir = r"C:\Users\tovst\Downloads\Курсовая работа\data\\"
    os.makedirs(output_dir, exist_ok=True)
    temp_file = os.path.join(output_dir, "tmdb_temp_checkpoint.csv")
    
    print(f"Запуск безопасной выгрузки за {start_year}–{end_year} гг. ({target_pages} страниц)...")
    
    for page in range(1, target_pages + 1):
        url = f"{BASE_URL}/discover/movie"
        params = {
            "include_adult": "false",
            "language": "ru-RU",
            "page": page,
            "primary_release_date.gte": f"{start_year}-01-01",
            "primary_release_date.lte": f"{end_year}-12-31",
            "sort_by": "popularity.desc"
        }
        
        success = False
        retries = 3
        
        # Цикл с попытками на случай временных сбоев или лимитов
        while retries > 0 and not success:
            response = requests.get(url, headers=HEADERS, params=params)
            
            if response.status_code == 200:
                data = response.json()
                results = data.get('results', [])
                if not results:
                    print("Достигнут конец доступных данных на сервере.")
                    return pd.DataFrame(all_movies)
                
                all_movies.extend(results)
                print(f"Страница {page}/{target_pages} успешно загружена (фильмов собрано: {len(all_movies)})")
                success = True
                
            elif response.status_code == 429:
                # Превышен лимит запросов (Rate Limit)
                print(f"Превышен лимит запросов (429). Ожидание 10 секунд перед повтором...")
                time.sleep(10)
                retries -= 1
            else:
                print(f"Ошибка HTTP {response.status_code} на странице {page}. Пробуем снова...")
                time.sleep(3)
                retries -= 1
        
        if not success:
            print(f"Не удалось загрузить страницу {page} после нескольких попыток. Прерываем цикл.")
            break 
        # Пауза между запросами, чтобы не нагружать API (около 3-4 запросов в секунду)
        time.sleep(0.35)
        
        # Промежуточное сохранение каждые 20 страниц, чтобы не потерять данные
        if page % 20 == 0:
            pd.DataFrame(all_movies).to_csv(temp_file, index=False, encoding='utf-8')
            print("  [Чекпоинт] Промежуточные данные сохранены на диск.")

    df = pd.DataFrame(all_movies)
    return df

# выгрузка
large_tmdb_df = fetch_tmdb_movies_safe(start_year=2010, end_year=2025, target_pages=75)

final_path = r"C:\Users\tovst\Downloads\Курсовая работа\data\tmdb_mass_dataset_2010_2025.csv"
large_tmdb_df.to_csv(final_path, index=False, encoding='utf-8')
print(f"\nВыгрузка завершена! Итоговый датасет сохранен: {final_path}")
print(f"Всего строк: {len(large_tmdb_df):,}")
display(large_tmdb_df[['title', 'release_date', 'popularity', 'vote_average']].head(3))

Запуск безопасной выгрузки за 2010–2025 гг. (75 страниц)...
Страница 1/75 успешно загружена (фильмов собрано: 20)
Страница 2/75 успешно загружена (фильмов собрано: 40)
Страница 3/75 успешно загружена (фильмов собрано: 60)
Страница 4/75 успешно загружена (фильмов собрано: 80)
Страница 5/75 успешно загружена (фильмов собрано: 100)
Страница 6/75 успешно загружена (фильмов собрано: 120)
Страница 7/75 успешно загружена (фильмов собрано: 140)
Страница 8/75 успешно загружена (фильмов собрано: 160)
Страница 9/75 успешно загружена (фильмов собрано: 180)
Страница 10/75 успешно загружена (фильмов собрано: 200)
Страница 11/75 успешно загружена (фильмов собрано: 220)
Страница 12/75 успешно загружена (фильмов собрано: 240)
Страница 13/75 успешно загружена (фильмов собрано: 260)
Страница 14/75 успешно загружена (фильмов собрано: 280)
Страница 15/75 успешно загружена (фильмов собрано: 300)
Страница 16/75 успешно загружена (фильмов собрано: 320)
Страница 17/75 успешно загружена (фильмов собрано: 340)
С

,title,release_date,popularity,vote_average
0,Обитель зла: Раккун-Сити,2021-11-24,166.2261,5.946
1,Зверополис 2,2025-11-26,126.2556,7.667
2,Мстители: Война бесконечности,2018-04-25,142.7426,8.200


### Характеристика датасета

1. Количественные характеристики:
- popularity — индекс популярности фильма в системе TMDB (динамический показатель просмотров и запросов).
- vote_average — средняя оценка фильма пользователями (по шкале от 0 до 10).
- vote_count — общее количество голосов, на основе которых выведена средняя оценка (показатель статистической надежности рейтинга).
- id — уникальный числовой идентификатор фильма (используется как ключевой атрибут для реляционного соединения с другими таблицами).
- budget (в расширенной выгрузке) — бюджет картины в долларах.

- revenue (в расширенной выгрузке) — мировые кассовые сборы.

2. Качественные (категориальные и текстовые) характеристики:
- title — официальное название фильма на языке запроса.
- original_title — оригинальное название фильма на языке страны-производителя.
- overview — синопсис или краткое текстовое описание сюжета (используется для NLP-анализа).
- original_language — двухбуквенный код языка оригинала (например, en, fr, ko).
- genre_ids — массив числовых идентификаторов жанров, к которым относится фильм.
- backdrop_path / poster_path — ссылки на медиафайлы графических ассетов (постеры и фоны).
- adult — булевый флаг (True/False), указывающий на наличие контента для взрослых.
- video — признак наличия видеоматериалов/трейлеров в базе.

3. Временные характеристики (Temporal Features)
- release_date — точная дата релиза фильма в формате ГГГГ-ММ-ДД. Ключевое поле для фильтрации и валидации датасета в строгом целевом временном диапазоне 2010–2025 годов в соответствии с требованиями ТЗ.



In [8]:
df_tmdb_mass = pd.read_csv(r"C:\Users\tovst\Downloads\Курсовая работа\data\tmdb_mass_dataset_2010_2025.csv")
summary_df = pd.DataFrame({
    'Тип данных': df_tmdb_mass.dtypes,
    'Количество пропусков': df_tmdb_mass.isnull().sum(),
    'Процент пропусков, %': (df_tmdb_mass.isnull().mean() * 100).round(2)
})
display(summary_df)
display(df_tmdb_mass.describe())

,Тип данных,Количество пропусков,"Процент пропусков, %"
adult,bool,0,0.00
backdrop_path,str,6,0.40
genre_ids,str,0,0.00
id,int64,0,0.00
title,str,0,0.00
original_language,str,0,0.00
original_title,str,0,0.00
overview,str,45,3.00
popularity,float64,0,0.00
poster_path,str,1,0.07


,id,popularity,vote_average,vote_count
count,1.500000e+03,1500.000000,1500.000000,1500.000000
mean,5.329274e+05,28.948672,6.782908,4827.533333
std,3.825622e+05,17.571968,0.821062,5614.484546
min,1.890000e+02,7.915000,0.000000,0.000000
25%,2.386308e+05,16.847175,6.304000,922.750000
50%,4.542895e+05,23.388400,6.800000,2966.000000
75%,8.116822e+05,35.293325,7.306000,6735.250000
max,1.608457e+06,166.226100,9.000000,41491.000000


## Слияние IMDB и TMDB

In [58]:
print("1. Загрузка подготовленных Parquet-датасетов...")
imdb_path = r"C:\Users\tovst\Downloads\Курсовая работа\data\imdb\imdb_merged_2010_2025.parquet"
tmdb_path = r"C:\Users\tovst\Downloads\Курсовая работа\data\tmdb\tmdb_movies_2010_2025.parquet"

imdb_df = pd.read_parquet(imdb_path)
tmdb_df = pd.read_parquet(tmdb_path)

print("\n2. Нормализация ключей слияния")
imdb_df['join_title'] = imdb_df['primaryTitle'].astype(str).str.strip().str.lower()
tmdb_df['join_title'] = tmdb_df['title'].astype(str).str.strip().str.lower()

imdb_df['join_year'] = imdb_df['startYear'].astype(int)
tmdb_df['join_year'] = tmdb_df['release_year'].astype(int)

merged_master_df = pd.merge(
    imdb_df,
    tmdb_df[['join_title', 'join_year', 'budget', 'revenue', 'popularity', 'overview']],
    on=['join_title', 'join_year'],
    how='left')

print(f"Объединено проектов на пересечении источников(с сохранением всех записей imbd): {len(merged_master_df):,}")
output_dir = r"C:\Users\tovst\Downloads\Курсовая работа\data\\"
os.makedirs(output_dir, exist_ok=True)
master_parquet_path = os.path.join(output_dir, "film_success_master_dataset(full_imdb).parquet")

merged_master_df.to_parquet(master_parquet_path, index=False)
print(f"Мастер-витрина сохранена: {master_parquet_path}")
print(f"Итоговый размер файла: {os.path.getsize(master_parquet_path) / (1024 * 1024):.2f} МБ")

display(merged_master_df[['primaryTitle', 'startYear', 'averageRating', 'numVotes', 'budget', 'revenue']].head(3))

1. Загрузка подготовленных Parquet-датасетов...

2. Нормализация ключей слияния
Объединено проектов на пересечении источников(с сохранением всех записей imbd): 160,825
Мастер-витрина сохранена: C:\Users\tovst\Downloads\Курсовая работа\data\\film_success_master_dataset(full_imdb).parquet
Итоговый размер файла: 10.97 МБ


,primaryTitle,startYear,averageRating,numVotes,budget,revenue
0,The Tango of the Widower and Its Distorting Mi...,2020,6.3,300,NaN,NaN
1,Appunti per un romanzo sull'immondezza,2022,6.9,41,NaN,NaN
2,The Other Side of the Wind,2018,6.6,8641,NaN,NaN


## Слияние rotten_tomatoes_movies и rotten_romatoes_reviews

In [60]:
data_dir = r"C:\Users\tovst\Downloads\Курсовая работа\data\rotten_tomaroes_reviews\\"
rt_movies = pd.read_csv(data_dir + "rotten_tomatoes_movies.csv")
rt_reviews = pd.read_csv(data_dir + "rotten_tomatoes_movie_reviews.csv")

In [66]:
# id_column = 'id' if 'id' in rt_movies.columns else 'movie_id'

# merged_rt = pd.merge(
#     rt_reviews,
#     rt_movies,
#     on=id_column,
#     how='inner',
#     suffixes=('_review', '_movie')
# )
# print(f"Объединенный датасет рецензий и фильмов: {len(merged_rt)}")

merged_rt['release_year'] = pd.to_datetime(merged_rt['releaseDateTheaters'], errors='coerce').dt.year
mask_nan = merged_rt['release_year'].isna()
merged_rt.loc[mask_nan, 'release_year'] = pd.to_datetime(merged_rt.loc[mask_nan, 'releaseDateStreaming'], errors='coerce').dt.year

print(f"Строк с успешно определенным годом релиза: {merged_rt['release_year'].notnull().sum():,}")

print("\n3. Фильтрация по целевому диапазону (2010–2025 гг.)...")
rt_filtered = merged_rt[
    (merged_rt['release_year'] >= 2010) & 
    (merged_rt['release_year'] <= 2025)
].copy()

print(f"Записей после фильтрации за 2010–2025 гг.: {len(rt_filtered):,}")

rt_output_dir = r"C:\Users\tovst\Downloads\Курсовая работа\data\rt"
os.makedirs(rt_output_dir, exist_ok=True)
rt_parquet_path = os.path.join(rt_output_dir, "rt_clean_2010_2025.parquet")

rt_filtered.to_parquet(rt_parquet_path, index=False)
print(f"Файл успешно сохранен: {rt_parquet_path}")
print(f"Размер файла: {os.path.getsize(rt_parquet_path) / (1024 * 1024):.2f} МБ")

Строк с успешно определенным годом релиза: 1,397,988

3. Фильтрация по целевому диапазону (2010–2025 гг.)...
Записей после фильтрации за 2010–2025 гг.: 828,464
Файл успешно сохранен: C:\Users\tovst\Downloads\Курсовая работа\data\rt\rt_clean_2010_2025.parquet
Размер файла: 112.96 МБ


## Слияние oskar и мастер-витрины

In [72]:
oscar_df['clean_year'] = pd.to_numeric(oscar_df['Year'].str.extract(r'(\d{4})')[0], errors='coerce')

# Фильтруем по периоду 2010–2025 гг.
oscar_filtered = oscar_df[
    (oscar_df['clean_year'] >= 2010) & 
    (oscar_df['clean_year'] <= 2025)
].copy()
print(f"Записей после фильтрации за 2010–2025 гг.: {len(oscar_filtered):,}")

# Преобразуем поле Winner в удобный формат (1 если победил, 0 если номинант)
oscar_filtered['is_winner'] = oscar_filtered['Winner'].notnull().astype(int)

oscar_output_dir = r"C:\Users\tovst\Downloads\Курсовая работа\data\the_oscar_award"
oscas_parquet_path = os.path.join(oscar_output_dir, "oscars_clean_2010_2025.parquet")
oscar_filtered.to_parquet(oscas_parquet_path, index=False)
print(f"Файл успешно сохранен: {oscas_parquet_path}")
print(f"Размер файла: {os.path.getsize(oscas_parquet_path) / (1024 * 1024):.2f} МБ")

Записей после фильтрации за 2010–2025 гг.: 2,049
Файл успешно сохранен: C:\Users\tovst\Downloads\Курсовая работа\data\the_oscar_award\oscars_clean_2010_2025.parquet
Размер файла: 0.20 МБ


In [74]:
master_path = r"C:\Users\tovst\Downloads\Курсовая работа\data\film_success_master_dataset(full_imdb).parquet"
oscar_parquet_path = r"C:\Users\tovst\Downloads\Курсовая работа\data\the_oscar_award\oscars_clean_2010_2025.parquet"

master_df = pd.read_parquet(master_path)
oscar_df = pd.read_parquet(oscar_parquet_path)
print(f"Строк в мастер-витрине: {len(master_df):,}")
print(f"Строк в датасете oskar_award: {len(oscar_df):,}")

oscar_df['join_title'] = oscar_df['Film'].astype(str).str.strip().str.lower()
oscar_df['join_year'] = oscar_df['clean_year'].astype(int)

oscar_agg = oscar_df.groupby(['join_title', 'join_year']).agg(
    oscar_nominations=('CanonicalCategory', 'count'),
    oscar_wins=('is_winner', 'sum')
).reset_index()

print(f"Уникальных фильмов-номинантов на Оскар (2010–2025): {len(oscar_agg):,}")

master_df = pd.merge(
    master_df,
    oscar_agg,
    on=['join_title', 'join_year'],
    how='left'
)


master_df['oscar_nominations'] = master_df['oscar_nominations'].fillna(0).astype(int)
master_df['oscar_wins'] = master_df['oscar_wins'].fillna(0).astype(int)

master_df.to_parquet(master_path, index=False)
print(f"Мастер-витрина обновлена: {master_path}")
print(f"Итоговое количество колонок в витрине: {len(master_df.columns)}")
#фильмы в выборке, которые получили Оскар
display(master_df[master_df['oscar_wins'] > 0][['primaryTitle', 'startYear', 'oscar_nominations', 'oscar_wins', 'revenue']].head(5))

Строк в мастер-витрине: 160,825
Строк в датасете oskar_award: 2,049
Уникальных фильмов-номинантов на Оскар (2010–2025): 839
Мастер-витрина обновлена: C:\Users\tovst\Downloads\Курсовая работа\data\film_success_master_dataset(full_imdb).parquet
Итоговое количество колонок в витрине: 23


,primaryTitle,startYear,oscar_nominations,oscar_wins,revenue
114,Toy Story 3,2010,5,2,1.066970e+09
125,Lincoln,2012,12,2,2.752934e+08
148,Life of Pi,2012,11,4,6.090166e+08
270,The Wolfman,2010,1,1,0.000000e+00
279,Dallas Buyers Club,2013,6,3,5.519828e+07


In [76]:
master_path = r"C:\Users\tovst\Downloads\Курсовая работа\data\film_success_master_dataset(full_imdb).parquet"
rt_path = r"C:\Users\tovst\Downloads\Курсовая работа\data\rt\rt_clean_2010_2025.parquet"

master_df = pd.read_parquet(master_path)
rt_df = pd.read_parquet(rt_path)

print(f"Строк в мастер-витрине: {len(master_df):,}")
print(f"Строк в очищенном Rotten Tomatoes: {len(rt_df):,}")

rt_df['join_title'] = rt_df['title'].astype(str).str.strip().str.lower()
rt_df['join_year'] = rt_df['release_year'].astype(int)

# Группируем отзывы и оценки по каждому фильму
rt_agg = rt_df.groupby(['join_title', 'join_year']).agg(
    rt_audience_score=('audienceScore', 'mean'),
    rt_tomatometer=('tomatoMeter', 'mean'),
    rt_reviews_count=('reviewId', 'count') if 'reviewId' in rt_df.columns else ('id', 'count'),
    rt_positive_ratio=('scoreSentiment', lambda x: (x == 'POSITIVE').mean() if 'scoreSentiment' in rt_df.columns else None)
).reset_index()

print(f"Уникальных фильмов в Rotten Tomatoes после агрегации: {len(rt_agg):,}")

print("\nВыполнение слияния с мастер-витриной (Left Join)...")
master_df = pd.merge(
    master_df,
    rt_agg,
    on=['join_title', 'join_year'],
    how='left'
)

print(f"Размер мастер-витрины после интеграции RT: {len(master_df):,}")
print(f"Фильмов с заполненным Tomatometer: {master_df['rt_tomatometer'].notnull().sum():,}")

master_df.to_parquet(master_path, index=False)
print(f"Файл успешно сохранен: {master_path}")

Строк в мастер-витрине: 160,825
Строк в очищенном Rotten Tomatoes: 828,464
Уникальных фильмов в Rotten Tomatoes после агрегации: 30,145

Выполнение слияния с мастер-витриной (Left Join)...
Размер мастер-витрины после интеграции RT: 160,825
Фильмов с заполненным Tomatometer: 6,984
Файл успешно сохранен: C:\Users\tovst\Downloads\Курсовая работа\data\film_success_master_dataset(full_imdb).parquet


## Аннотирование и векторизация

In [79]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
master_df['clean_overview'] = master_df['overview'].fillna("").astype(str)
print(f"Количество текстов для векторизации: {(master_df['clean_overview'] != '').sum():,}")

tfidf = TfidfVectorizer(max_features=1000, stop_words='english', lowercase=True)
tfidf_matrix = tfidf.fit_transform(master_df['clean_overview'])# Преобразуем текстовые синопсисы в матрицу числовых признаков

Количество текстов для векторизации: 1,336


In [81]:
# Проверим заполненность колонок Rotten Tomatoes в мастер-витрине
rt_cols = [col for col in master_df.columns if 'rt_' in col]
print("Колонки Rotten Tomatoes в витрине:", rt_cols)

if 'rt_reviews_count' in master_df.columns:
    print(f"Фильмов с отзывами из Rotten Tomatoes: {master_df['rt_reviews_count'].notnull().sum():,}")

Колонки Rotten Tomatoes в витрине: ['rt_audience_score', 'rt_tomatometer', 'rt_reviews_count', 'rt_positive_ratio']
Фильмов с отзывами из Rotten Tomatoes: 9,964


## Мастер-витрина

In [83]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 160825 entries, 0 to 160824
Data columns (total 28 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   tconst                 160825 non-null  str    
 1   titleType              160825 non-null  str    
 2   primaryTitle           160823 non-null  str    
 3   isAdult                160825 non-null  int64  
 4   startYear              160825 non-null  int32  
 5   runtimeMinutes         149146 non-null  float64
 6   genres                 158313 non-null  str    
 7   averageRating          160825 non-null  float64
 8   numVotes               160825 non-null  int64  
 9   num_actors             160825 non-null  int32  
 10  num_actresses          160825 non-null  int32  
 11  num_directors          160825 non-null  int32  
 12  directors              158413 non-null  str    
 13  writers                137328 non-null  str    
 14  release_regions_count  160693 non-null  float64

## Выбор структуры хранения данных

Для хранения, обработки и интеграции объединенного массива данных кинематографа в рамках пайплайна CRISP-DM был выбран колоночный формат Apache Parquet.
Выбор обусловлен следующими факторами:
- Колоночная организация хранения
  В отличие от построчных форматов (CSV, JSON), в Parquet данные каждого столбца хранятся отдельно,     что  важно для исследовательского анализа данных (EDA) и машинного обучения, так как позволяет        считывать из файла только те признаки, которые задействованы в текущем расчете, не загружая в         оперативную память весь массив ненужных текстовых полей.

- Эффективное встроенное сжатие
  Формат поддерживает продвинутые алгоритмы сжатия (такие как Snappy или Gzip) на уровне блоков. Это    снижает физический размер файлов на диске в 3–5 раз по сравнению с текстовыми CSV-файлами, что        оптимизирует дисковое пространство и ускоряет передачу данных.

- Строгая типизация и сохранение схемы
  CSV-формат не хранит информацию о типах данных, из-за чего при каждом повторном чтении числовые       поля (float64, int64) могут ошибочно интерпретироваться как строки, требуя ручного парсинга.          Parquet фиксирует типы данных и метаданные на этапе сохранения, полностью исключая ошибки типизации   при передаче датасета в модели машинного обучения.

- Оптимизация под аналитические запросы (OLAP)
  Формат разработан специально для аналитических нагрузок и беспрепятственно интегрируется со стеком    инструментов обработки данных, обеспечивая высокую скорость сериализации и десериализации больших     объемов структурированной и частично неструктурированной информации.